In [2]:
# thread: program에서 일을 처리하는 단위(많을수록 동시에 일 처리 가능)
# GIL: 한 번에 하나의 thread만 python code를 실행하게 만듦
# python은 GIL로 인해 thread가 GIL을 얻지 못하면 일을 수행할 수 없음
# 비동기 방식을 쓰면 한 작업의 대기 시간 발생 시 다음 작업이 바로 이어받을 수 있다.
# 대기 중인 작업은 OS가 처리하고, thread는 다른 작업을 실행한다.

In [3]:
# 비동기 함수
async def greet():
    return "hello"

# async만 있으면 coroutine 객체만 반환하고 실행은 안 됨
result = greet()
print(type(result))  # <class 'coroutine'> - hello가 출력되지 않음
# RuntimeWarning: coroutine 'greet' was never awaited

<class 'coroutine'>


In [ ]:
import asyncio

# await 
# 1. coroutine 실행
# 2. event loop가 해당 coroutine이 끝날 때까지 기다렸다가 결과를 반환
# ※ 이때 계속 thread를 잡고 있는 것이 아니라 eventloop를 거쳐 os에 맡김
async def fetch_data():
  print("요청 시작")
  await asyncio.sleep(2) # 여기서 양보
  print("응답 수신")
  return "data"
  
async def main():
  result = await fetch_data()
  print(result)

# 일반 python script로 실행하면 동작 잘 됨
# asyncio.run(main())
await main()

요청 시작
응답 수신
data


In [ ]:
# 동기: DB 기다리는 동안 thread 잠김 -> 10명 요청 시 한 명씩 줄줄이 밀림
# 비동기: DB 기다리는 동안 thread 양보 -> 10명 요청 거의 동시 처리 가능
# 즉, await는 "기다리는 동안 다른 요청도 처리해" 라는 의미

# async/await 없으면 (동기 방식)
# @app.get("/data")
# def get_data():  # 일반 함수
#     result = fetch_from_db()  # DB 응답 기다리는 동안 thread 완전히 잠김
#     return result

# async/await 있으면
# @app.get("/data")
# async def get_data():
#     result = await fetch_from_db()  # 기다리는 동안 양보!
#     return result

In [ ]:
# background
import asyncio

async def fetch_data(url):
  await asyncio.sleep(1)
  return f"{url} done"

async def main():
  # create_task로 background에 등록
  # 중간에 await가 없으면 event_loop가 비어지지 않으므로 계속해서 background에서 대기(main이 event_loop를 쥐고 있으므로)
  task = asyncio.create_task(fetch_data("api/1"))
  # task를 await로 실행하면 event_loop가 main을 놓기 때문에 event_loop가 create_task 안의 coroutine 객체를 background 실행으로 위임
  # 그 전에 await가 있으면 background를 실행시킬 수 있으므로 실행된 결과가 반환
  result = await task
  return result

await main()

'api/1 done'

In [27]:
# 동시 처리
import asyncio
async def task_a():
  print("A: start")
  await asyncio.sleep(1)
  print("A: end")

async def task_b():
  print("B:start")
  await asyncio.sleep(2)
  print("B:end")

async def main():
  # 동시 실행
  await asyncio.gather(task_a(), task_b())

await main()

A: start
B:start
A: end
B:end
